# 2025 DL Lab5: Object Detection on Pascal VOC

Before we start, please put **your name** and **SID** in following format: <br>
Hi I'm 陸仁賈, 314831000.

**Your Answer:**    
Hi I'm 宋尚勳, 314834014

## Overview

This project focuses on object detection using the Pascal VOC dataset. 

The goal is to identify and locate various objects within images by training and evaluating detection models.
 
The dataset provides annotated images across multiple categories, making it a standard benchmark for evaluating object detection performance.


## Kaggle Competition
Kaggle is an online community of data scientists and machine learning practitioners. Kaggle allows users to find and publish datasets, explore and build models in a web-based data-science environment, work with other data scientists and machine learning engineers, and enter competitions to solve data science challenges.

This assignment use kaggle to calculate your grade.  
Please use this [**LINK**](https://www.kaggle.com/t/3fd493e454a744bdacc7f2918f9a2605) to join the competition.

## Unzip Data

Unzip `dataset.zip` 

+ `vocall_test.txt` : list for the training set
+ `vocall_test.txt` : list for the validation set
+ `vocall_test.txt` : list for the test set
+ `image/` : contains all images.


The train set contains 8,218 images, the val set contains 3,823 images, and the test set contains 8,920 images.


#### You are allowed to use a **backbone model**, but only those available from the **timm package** (https://huggingface.co/timm/models).

# Import package

In [1]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
from torch.amp import autocast, GradScaler
from src.yolo import getODmodel
from yolo_loss import YOLOv3Loss
from src.dataset import VocDetectorDataset, train_data_pipelines, test_data_pipelines, collate_fn
from src.eval_voc import evaluate
from src.config import GRID_SIZES, ANCHORS
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR #1.1 新增

e:\project\deeplearning_HW5\Lab5\Lab5\src\dataset.py:17: UserWarning: Argument(s) 'variance_limit' are not valid for transform GaussNoise
  A.GaussNoise(variance_limit=(10.0, 50.0), p=0.5),


In [2]:
print("1. 設定超參數...")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
num_epochs = 70      # 延長訓練週期以獲得更好效果
batch_size = 16      
learning_rate = 1e-4 # 使用較低的學習率進行微調
warmup_epochs = 5    # 啟用 5 個週期的學習率預熱

# 損失函數權重
lambda_coord=5.0
lambda_obj=1.0
lambda_noobj=0.5
lambda_class=1.0
print(f"   - 學習率: {learning_rate}, 預熱週期: {warmup_epochs}, 總週期: {num_epochs}")

print("2. 初始化模型...")
load_network_path = None 
pretrained = True 
model = getODmodel(pretrained=pretrained).to(device)
print("   - 模型已成功建立並移至 device。")


print("3. 設定損失函數、優化器與排程器...")

criterion = YOLOv3Loss(lambda_coord, lambda_obj, lambda_noobj, lambda_class, ANCHORS).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=5e-4)

warmup_scheduler = LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs)

cosine_scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs - warmup_epochs, eta_min=1e-6)

lr_scheduler = SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])

use_amp = torch.cuda.is_available()
scaler = GradScaler(enabled=use_amp)
print("   - 所有訓練元件均已準備就緒！")

1. 設定超參數...
   - 學習率: 0.0001, 預熱週期: 5, 總週期: 70
2. 初始化模型...


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


   - 模型已成功建立並移至 device。
3. 設定損失函數、優化器與排程器...
   - 所有訓練元件均已準備就緒！


In [3]:
# Data paths
file_root_train = './dataset/image/'
annotation_file_train = './dataset/vocall_train.txt'
file_root_val = './dataset/image/'
annotation_file_val = './dataset/vocall_val.txt'
 # Data paths
file_root_train = './dataset/image/'
annotation_file_train = './dataset/vocall_train.txt'
file_root_val = './dataset/image/'
annotation_file_val = './dataset/vocall_val.txt'

# Create datasets
print('Loading datasets...')
train_dataset = VocDetectorDataset(
    root_img_dir=file_root_train,
    dataset_file=annotation_file_train,
    train=True,
    transform=train_data_pipelines,
    grid_sizes=GRID_SIZES,
    encode_target=True
)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=collate_fn,
    shuffle=True,
    num_workers=4,
)
print(f'Loaded {len(train_dataset)} train images')

val_dataset = VocDetectorDataset(
    root_img_dir=file_root_val,
    dataset_file=annotation_file_val,
    train=False,
    transform=test_data_pipelines,
    grid_sizes=GRID_SIZES,
    encode_target=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=collate_fn,
    shuffle=False,
    num_workers=4,
)
#for computing val maps
eval_dataset = VocDetectorDataset(
    root_img_dir=file_root_val,
    dataset_file=annotation_file_val,
    train=False,
    transform=test_data_pipelines,
    grid_sizes=GRID_SIZES,
    encode_target=False,
)
eval_loader = DataLoader(
    eval_dataset,
    batch_size=batch_size,
    collate_fn=collate_fn,
    shuffle=False,
    num_workers=4
)
print(f'Loaded {len(val_dataset)} val images')

Loading datasets...
Initializing dataset
Loaded 8218 train images
Initializing dataset
Initializing dataset
Loaded 3823 val images


## Initialization

### Only backbone model on timm is acceptable (https://huggingface.co/timm/models).
### You can modify model name in yolo class

### Training Loop

In [4]:
# Training loop
print('\nStarting training...')
torch.cuda.empty_cache()
best_val_loss = np.inf
for epoch in range(num_epochs):
    model.train()
    print(f'\n\nStarting epoch {epoch + 1} / {num_epochs}')
    for i, (images, target) in enumerate(train_loader):
        # Move to device
        images = images.to(device)
        target = [t.to(device) for t in target]
        # Forward pass
        optimizer.zero_grad()
        with autocast("cuda", enabled=use_amp):
            pred = model(images)
            # pred and target are lists of each scales
            loss_dict = criterion(pred, target)
        # Backward pass with mixed precision support
        scaler.scale(loss_dict['total']).backward()
        scaler.step(optimizer)
        scaler.update()
        # Print progress
        if i % 50 == 0:
            outstring = f'Epoch [{epoch+1}/{num_epochs}], Iter [{i+1}/{len(train_loader)}], Loss: '
            outstring += ', '.join(f"{key}={val :.3f}" for key, val in loss_dict.items())
            print(outstring)
    lr_scheduler.step()
    learning_rate = lr_scheduler.get_last_lr()[0]
    print(f'Learning Rate for this epoch: {learning_rate}')
    # Validation
    with torch.no_grad():
        val_loss = 0.0
        model.eval()
        for i, (images, target) in enumerate(val_loader):
            # Move to device
            images = images.to(device)
            target = [t.to(device) for t in target]
            # Forward pass
            pred = model(images)
            loss_dict = criterion(pred, target)
            val_loss += loss_dict['total'].item()

        val_loss /= len(val_loader)
        print(f'Validation Loss: {val_loss:.4f}')

    # Save best model
    if best_val_loss > val_loss:
        best_val_loss = val_loss
        print(f'Updating best val loss: {best_val_loss:.5f}')
        os.makedirs('checkpoints', exist_ok=True)
        torch.save(model.state_dict(), 'checkpoints/best_detector.pth')

    # Save checkpoint
    if (epoch + 1) in [5, 10, 20, 30, 40]:
        torch.save(model.state_dict(), f'checkpoints/detector_epoch_{epoch+1}.pth')

    torch.save(model.state_dict(), 'checkpoints/detector.pth')

    # Evaluate on val set
    if (epoch + 1) % 5 == 0:
        print('\nEvaluating on validation set...')
        val_aps = evaluate(model, eval_loader)
        print(f'Epoch {epoch}, mAP: {np.mean(val_aps):.4f}')


Starting training...


Starting epoch 1 / 70
Epoch [1/70], Iter [1/514], Loss: total=19.077, box=0.733, obj=0.724, noobj=0.761, cls=14.310
Epoch [1/70], Iter [51/514], Loss: total=17.795, box=0.672, obj=0.656, noobj=0.763, cls=13.399
Epoch [1/70], Iter [101/514], Loss: total=16.741, box=0.662, obj=0.576, noobj=0.766, cls=12.475
Epoch [1/70], Iter [151/514], Loss: total=15.778, box=0.660, obj=0.540, noobj=0.768, cls=11.554
Epoch [1/70], Iter [201/514], Loss: total=15.895, box=0.741, obj=0.502, noobj=0.770, cls=11.302
Epoch [1/70], Iter [251/514], Loss: total=13.921, box=0.609, obj=0.440, noobj=0.772, cls=10.053
Epoch [1/70], Iter [301/514], Loss: total=13.518, box=0.596, obj=0.431, noobj=0.775, cls=9.720
Epoch [1/70], Iter [351/514], Loss: total=13.059, box=0.663, obj=0.421, noobj=0.776, cls=8.933
Epoch [1/70], Iter [401/514], Loss: total=11.387, box=0.562, obj=0.332, noobj=0.779, cls=7.857
Epoch [1/70], Iter [451/514], Loss: total=11.194, box=0.594, obj=0.315, noobj=0.784, cls=7.516
E

e:\project\deeplearning_HW3\deeplearning\Lib\site-packages\torch\optim\lr_scheduler.py:209: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


Learning Rate for this epoch: 0.0001
Validation Loss: 3.7912
Updating best val loss: 3.79116

Evaluating on validation set...
---Evaluate model on validation samples---


100%|██████████| 239/239 [05:49<00:00,  1.46s/it]


---class aeroplane ap 0.2687196870274176---
---class bicycle ap 0.22884746987096616---
---class bird ap 0.10950749114266971---
---class boat ap 0.030013751903460787---
---class bottle ap 0.04568387003367287---
---class bus ap 0.39895166663283355---
---class car ap 0.12122499132835772---
---class cat ap 0.501189283758583---
---class chair ap 0.09055031207254848---
---class cow ap 0.0931679365019588---
---class diningtable ap 0.08970831538151766---
---class dog ap 0.4059838901775069---
---class horse ap 0.30642706878273535---
---class motorbike ap 0.23579168861629035---
---class person ap 0.2507951837625487---
---class pottedplant ap 0.06485416501881305---
---class sheep ap 0.029433628754168932---
---class sofa ap 0.35890700898299466---
---class train ap 0.29886839512633245---
---class tvmonitor ap 0.17986953065521843---
---map 0.20542476677652974---
Epoch 4, mAP: 0.2054


Starting epoch 6 / 70
Epoch [6/70], Iter [1/514], Loss: total=4.002, box=0.362, obj=0.085, noobj=0.371, cls=1.923
Ep

...

---class aeroplane ap 0.5645909151424844---
---class bicycle ap 0.6314382035445626---
---class bird ap 0.5970092415165749---
---class boat ap 0.24905265009336416---
---class bottle ap 0.23227908584829615---
---class bus ap 0.6589820718112926---
---class car ap 0.47643057426667734---
---class cat ap 0.8858047781046023---
---class chair ap 0.38696801670224373---
---class cow ap 0.46277586094412426---
---class diningtable ap 0.6065084585521823---
---class dog ap 0.781490341200647---
---class horse ap 0.7098443315909382---
---class motorbike ap 0.7699342410143386---
---class person ap 0.5371459632481288---
---class pottedplant ap 0.35281794517363496---
---class sheep ap 0.4637241652402938---
---class sofa ap 0.5711556009959893---
---class train ap 0.7393329547798861---
---class tvmonitor ap 0.6490118832496579---
---map 0.5663148641509961---
Epoch 59, mAP: 0.5663


Starting epoch 61 / 70
Epoch [61/70], Iter [1/514], Loss: total=0.537, box=0.097, obj=0.007, noobj=0.079, cls=0.005
Epoch [61/7

100%|██████████| 239/239 [01:28<00:00,  2.69it/s]


---class aeroplane ap 0.597714648047543---
---class bicycle ap 0.6477293371189756---
---class bird ap 0.6604195612177962---
---class boat ap 0.2862923965676159---
---class bottle ap 0.24984547872246993---
---class bus ap 0.663068810392403---
---class car ap 0.45589735714818624---
---class cat ap 0.8997670458928072---
---class chair ap 0.3853173892944548---
---class cow ap 0.5015171164633503---
---class diningtable ap 0.6132646737786857---
---class dog ap 0.7881317919500452---
---class horse ap 0.7305015273464253---
---class motorbike ap 0.7880007813926451---
---class person ap 0.5734311287353921---
---class pottedplant ap 0.3473367832155861---
---class sheep ap 0.4708464839460912---
---class sofa ap 0.5998920882036143---
---class train ap 0.7613234229048063---
---class tvmonitor ap 0.6580203582426827---
---map 0.5839159090290788---
Epoch 64, mAP: 0.5839


Starting epoch 66 / 70
Epoch [66/70], Iter [1/514], Loss: total=0.584, box=0.096, obj=0.017, noobj=0.081, cls=0.044
Epoch [66/70], I

100%|██████████| 239/239 [01:28<00:00,  2.69it/s]


---class aeroplane ap 0.5739407984609811---
---class bicycle ap 0.6306175002338106---
---class bird ap 0.6668037041007421---
---class boat ap 0.2640980167697146---
---class bottle ap 0.24939095623357083---
---class bus ap 0.6343698372693358---
---class car ap 0.456576478054904---
---class cat ap 0.8969242672703566---
---class chair ap 0.39818987785500837---
---class cow ap 0.48785646729584975---
---class diningtable ap 0.6122512797595869---
---class dog ap 0.7931412346527145---
---class horse ap 0.7125592637052123---
---class motorbike ap 0.7741863614501274---
---class person ap 0.552659137986571---
---class pottedplant ap 0.3440466101086047---
---class sheep ap 0.4841012404367396---
---class sofa ap 0.5889063992774747---
---class train ap 0.7379127397648935---
---class tvmonitor ap 0.6610155641490776---
---map 0.5759773867417639---
Epoch 69, mAP: 0.5760


# Kaggle submission

### Predict Result

Predict the results based on testing set. Upload to [Kaggle](https://www.kaggle.com/t/3fd493e454a744bdacc7f2918f9a2605).

**How to upload**

1. Click the folder icon in the left hand side of Colab.
2. Right click "result.csv". Select "Download"
3. To kaggle. Click "Submit Predictions"
4. Upload the result.csv
5. System will automaticlaly calculate the accuracy of 50% dataset and publish this result to leaderboard.


In [5]:
!python predict_test.py

e:\project\deeplearning_HW5\Lab5\Lab5\src\dataset.py:17: UserWarning: Argument(s) 'variance_limit' are not valid for transform GaussNoise
  A.GaussNoise(variance_limit=(10.0, 50.0), p=0.5),
